# セミナーII 第2回 実践：ViT のパッチ特徴で画像の部分を比べる（DINOv2）

ViT は、画像を 14×14 画素などのパッチに分け、**パッチごとに特徴ベクトル**を出力します。**DINOv2** は、ラベルも文章も使わずに、大量の画像だけで学習した ViT です。そのパッチの特徴ベクトルは、写っている物の「部分」をよく表していて、学習し直さなくても、次のようなことに使えます。

- 画像の中で、似た部分（同じ種類の部分）を探す
- 物と背景を分ける
- 別の画像の中で、対応する部分（猫の目と猫の目など）を見つける

Depth Anything V2 の画像エンコーダも DINOv2 です。CLIP は画像全体を1本のベクトルにしましたが、ここではパッチごとのベクトルを使います。

**最初に**

1. メニューの「ファイル」→「ドライブにコピーを保存」で、自分のコピーを作ります。
2. 「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選びます。
3. 上のセルから順に実行します（セルの左の ▶、または **Shift + Enter**）。

ログインや Google ドライブへのアクセスの許可は必要ありません。全体で約35分（うち計算 約3分）です。

- セルの右に入力欄やスライダーが出るセルでは、そこで値を選んでから実行します（コードを書き換えても同じです）。位置は、画像の左上を (0, 0)、右下を (1, 1) とした数で指定します。
- 図を描く関数のセルは、コードを畳んであります。実行するだけでかまいません。
- しばらく操作しないと接続が切れます。`NameError` が出たときは、1. から順に実行し直してください。

## 0. このノートブックの流れ

目安：約1分

| 節 | やること |
| --- | --- |
| 1. 準備 | ライブラリとモデルを読み込む |
| 2. 手法を知る | ViT が出力するパッチの特徴ベクトル。DINOv2 の学習のしかた |
| 3. 例の画像で試す | パッチの特徴ベクトルを主成分分析で3次元にして、色で見る |
| 4. 自分の画像で試す | 自分の写真で同じことをする |
| 5. やってみよう | 似たパッチを探す、入力の大きさを変える |
| 6. 活用 | 前景を切り出す、別の画像と対応する部分を見つける、少ない枚数で分類する |
| 7. まとめ | 分かったこと |

## 1. 準備

目安：約3分（うち計算 約1分）

`device: cuda` と GPU の名前が表示されれば準備完了です。`device: cpu` と表示されたら、「ランタイム」→「ランタイムのタイプを変更」で GPU を選んで、もう一度実行してください。

In [ ]:
import importlib.util
import io
import subprocess
import sys
import time

# Colab に入っていないライブラリがあれば入れる
for package in ["transformers", "datasets", "pooch"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from PIL import Image, ImageOps
from skimage import data
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32  # GPU では半分の精度で速く計算する
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

DINOv2 の Base（約8700万パラメータ）を読み込み、パッチの特徴ベクトルを取り出す関数と、主成分分析の関数を定義します。初回は重み（約350MB）をダウンロードします。`HF_TOKEN` に関する警告が表示されても無視してかまいません。

- モデル：[facebook/dinov2-base](https://huggingface.co/facebook/dinov2-base)（Apache-2.0）
- 論文：M. Oquab et al., "DINOv2: Learning Robust Visual Features without Supervision," Transactions on Machine Learning Research, 2024（[arXiv:2304.07193](https://arxiv.org/abs/2304.07193)）

In [ ]:
DINO_ID = "facebook/dinov2-base"
processor = AutoImageProcessor.from_pretrained(DINO_ID)
model = AutoModel.from_pretrained(DINO_ID, dtype=dtype).to(device).eval()
PATCH = model.config.patch_size  # 14


def extract(image, short_side=448):
    # 画像 → パッチの特徴ベクトル（縦のパッチ数 × 横のパッチ数 × 768）と、画像全体のベクトル（768）
    w, h = image.size
    scale = short_side / min(w, h)
    size = {"height": round(h * scale / PATCH) * PATCH, "width": round(w * scale / PATCH) * PATCH}  # 14 の倍数
    pixel_values = processor(images=image.convert("RGB"), size=size, do_center_crop=False,
                             return_tensors="pt")["pixel_values"].to(device, dtype)
    with torch.no_grad():
        output = model(pixel_values=pixel_values)
    gh, gw = pixel_values.shape[-2] // PATCH, pixel_values.shape[-1] // PATCH
    tokens = output.last_hidden_state[0].float().cpu()  # (1 + gh*gw, 768)。先頭は CLS トークン
    return {"input": tuple(pixel_values.shape), "tokens": tuple(tokens.shape),
            "patches": tokens[1:].reshape(gh, gw, -1), "cls": output.pooler_output[0].float().cpu()}


def pca(x, k=3):
    # 主成分分析：(個数, 次元) → 平均を引いて特異値分解し、分散の大きい k 方向に射影する → (個数, k)
    x = x - x.mean(dim=0)
    _, s, vt = torch.linalg.svd(x, full_matrices=False)
    ratio = (s[:k] ** 2 / (s ** 2).sum()).numpy()  # 各主成分が説明する分散の割合
    return x @ vt[:k].T, ratio


def to_rgb(x):
    # 各列を 0〜1 にそろえて、色（R, G, B）として使う
    low, high = x.min(dim=0).values, x.max(dim=0).values
    return ((x - low) / (high - low + 1e-8)).numpy()


print(f"パラメータの数: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M、パッチの大きさ: {PATCH}×{PATCH} 画素、"
      f"特徴ベクトルの次元: {model.config.hidden_size}")

図を描く関数と、例の画像を用意します。

In [ ]:
# @title 図を描く関数と、例の画像（実行するだけでよい） { display-mode: "form" }
PETS_ID = "timm/oxford-iiit-pet"  # 第1回と同じデータセット（6-2、6-3 で使う）
PETS_REVISION = "089695c834a7deb60505b7cc506672db1c31a6aa"
_pets = {}

images = {  # scikit-image に入っている写真
    "cat": Image.fromarray(data.chelsea()),                      # 撮影 Stefan van der Walt（CC0）
    "astronaut": Image.fromarray(data.astronaut()),              # NASA（パブリックドメイン）
    "coffee": Image.fromarray(data.coffee()),                    # 撮影 Rachel Michetti（CC0）
    "motorcycle": Image.fromarray(data.stereo_motorcycle()[0]),  # Middlebury 2014 ステレオデータセット
}


def show_pca(image, feats, title=""):
    # 入力、第1主成分、第1〜3主成分を色にしたものを並べる
    gh, gw, d = feats["patches"].shape
    components, ratio = pca(feats["patches"].reshape(-1, d))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    axes[0].imshow(image)
    axes[0].set_title(title)
    axes[1].imshow(components[:, 0].reshape(gh, gw).numpy(), cmap="coolwarm")
    axes[1].set_title(f"1st principal component ({ratio[0]:.0%} of variance)")
    axes[2].imshow(to_rgb(components).reshape(gh, gw, 3))
    axes[2].set_title(f"PC1-3 as RGB ({gh}x{gw} patches)")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def patch_at(feats, x, y):
    # 位置 (x, y)（0〜1）にあるパッチの番号（行, 列）
    gh, gw, _ = feats["patches"].shape
    return min(int(y * gh), gh - 1), min(int(x * gw), gw - 1)


def similarity_map(feats, vector):
    # ベクトルと、各パッチの特徴ベクトルとのコサイン類似度（縦のパッチ数 × 横のパッチ数）
    return F.cosine_similarity(feats["patches"], vector[None, None, :], dim=-1)


def foreground(feats_list):
    # 第1主成分で物と背景を分ける（複数の画像は、パッチをまとめて主成分分析する）
    # 画像の縁のパッチが多い側を背景とみなして符号をそろえ、各画像のマスク（縦 × 横）のリストを返す
    shapes = [f["patches"].shape[:2] for f in feats_list]
    first = pca(torch.cat([f["patches"].reshape(-1, f["patches"].shape[-1]) for f in feats_list]), k=1)[0][:, 0]
    maps = list(torch.split(first, [h * w for h, w in shapes]))
    maps = [m.reshape(h, w) for m, (h, w) in zip(maps, shapes)]
    border = torch.cat([torch.cat([m[0], m[-1], m[:, 0], m[:, -1]]) for m in maps])
    sign = -1 if (border > 0).float().mean() > 0.5 else 1
    return [sign * m > 0 for m in maps]


def load_pets():
    # 第1回のデータセット（train と test）。1回だけ読み込んで使い回す
    if not _pets:
        start = time.time()
        dataset = load_dataset(PETS_ID, revision=PETS_REVISION)
        _pets["train"], _pets["test"] = dataset["train"], dataset["test"]
        _pets["names"] = [n.replace("_", " ") for n in dataset["train"].features["label"].names]
        print(f"第1回のデータセットを読み込みました（{time.time() - start:.0f} 秒）")
    return _pets


def fit_size(image, max_side=1024):
    # 大きすぎる写真は縮小する
    image = image.convert("RGB")
    scale = max_side / max(image.size)
    if scale < 1:
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.LANCZOS)
    return image


print("例の画像:", list(images))

## 2. 手法を知る

目安：約7分

### 2-1. ViT が出力するもの

ViT は、画像を 14×14 画素のパッチに分け、各パッチを1つのトークン（ベクトル）にします。先頭に画像全体を表す **CLS トークン**を足して Transformer ブロックを重ね、最後の層で**トークンごとに768次元の特徴ベクトル**を出力します。

| 出力 | 形 | これまでの例 |
| --- | --- | --- |
| CLS トークンのベクトル | (768,) | CLIP の画像のベクトル、第1回の分類ヘッドの入力（画像全体で1本） |
| パッチの特徴ベクトル | (縦のパッチ数, 横のパッチ数, 768) | 第1回の CNN の特徴マップ（7×7×512）にあたる。Depth Anything V2 は、これを画素ごとの奥行きに変換していた |

アテンションによって、各パッチの特徴ベクトルには、周りのパッチの情報も混ざっています。そのため「この画素の色」ではなく、「猫の目」「カップの縁」のような、**物の部分としての意味**を表すベクトルになります。

下のセルで、猫の画像からパッチの特徴ベクトルを取り出し、形を確かめます。

In [ ]:
feats = extract(images["cat"])
print("モデルへの入力 (枚数, 色, 縦, 横):", feats["input"])
print("最後の層の出力 (トークン数, 次元):", feats["tokens"], "← 1 (CLS) + パッチ数")
print("パッチの特徴ベクトル (縦, 横, 次元):", tuple(feats["patches"].shape))
print("画像全体のベクトル (次元,):", tuple(feats["cls"].shape))

448×672 の入力なら、パッチは 32×48 = 1536 個です。1536 個のパッチのそれぞれが、768次元のベクトルを持っています。

### 2-2. DINOv2 の学習（ラベルなしの自己教師あり学習）

DINOv2 は、**ラベルも文章も使わずに**、画像だけで学習しています（**自己教師あり学習**）。

- **データ**：Web から集めた画像のうち、ImageNet などの整ったデータセットの画像に似たものを選んで作った、1億4200万枚（LVD-142M）
- **生徒と教師**：同じ画像から切り出した、大きさや位置の違う複数の画像を、2つのネットワーク（生徒と教師）に入れる。生徒の出力が教師の出力に近づくように、生徒の重みを更新する。教師の重みは、生徒の重みの移動平均にする
  - 画像全体（CLS トークン）で、同じ画像の別の切り出しどうしの出力をそろえる（DINO の Loss）
  - 一部のパッチを隠した画像を生徒に入れ、隠したパッチの出力を、隠していない画像を見た教師の出力に近づける（iBOT の Loss）
- **大きさ**：一番大きいモデル（ViT-g、11億パラメータ）を学習し、それを手本にして、小さいモデル（S、B、L）を学習（蒸留）している

「同じ画像の別の切り出しは、同じ物を表す」「隠した部分は、周りから推測できる」ことを手がかりにして、正解のラベルなしで、物とその部分を区別する特徴を身につけています。

## 3. 例の画像で試す

目安：約5分

パッチの特徴ベクトルは768次元なので、そのままでは見られません。そこで**主成分分析**（PCA）で、分散の大きい3つの方向に射影して3次元にし、それを色（R, G, B）として表示します。

- (1536, 768) → 主成分分析 → (1536, 3) → 色の画像 (32, 48, 3)
- 特徴ベクトルが似ているパッチは、似た色になる

中央の図は第1主成分（一番分散の大きい方向）だけを表示したもので、赤と青の境目が、物と背景の境目になっていることが多いです（6-1 で使います）。

In [ ]:
feats = extract(images["cat"])
gh, gw, d = feats["patches"].shape
components, ratio = pca(feats["patches"].reshape(-1, d))  # (1536, 768) → (1536, 3)
print(f"主成分分析: {(gh * gw, d)} → {tuple(components.shape)}")
print("各主成分が説明する分散の割合:", ratio.round(3))

for name, image in images.items():
    show_pca(image, extract(image), name)

**見どころ**：色の画像では、猫の目、鼻、耳、毛の部分ごとに色が分かれています。宇宙飛行士では、顔、宇宙服、ヘルメット、旗が、別々の色になっています。DINOv2 は、どれも部分の名前を教わっていません。それでも、特徴ベクトルが「物の部分」ごとにまとまっています。

## 4. 自分の画像で試す

目安：約5分（画像を用意する時間は除く）

`USE_UPLOAD` にチェックを入れて実行すると、アップロードのボタンが表示されます（Colab のみ）。アップロードした画像は、ファイル名（拡張子を除く）の名前で、5. と 6. でも使えます。

In [ ]:
USE_UPLOAD = False  # @param {type:"boolean"}

if USE_UPLOAD:
    from google.colab import files

    for filename, content in files.upload().items():
        name = filename.rsplit(".", 1)[0]
        images[name] = fit_size(Image.open(io.BytesIO(content)))
        show_pca(images[name], extract(images[name]), name)
print("使える画像の名前:", list(images))

## 5. やってみよう

目安：約8分

### 5-1. 似たパッチを探す

画像の中の1つのパッチ（赤い十字）を選び、そのパッチの特徴ベクトルと、すべてのパッチの特徴ベクトルのコサイン類似度を計算して、ヒートマップで表示します。明るいほど似ています。

例では、猫の右目（画像の左側の目）を選んでいます。もう一方の目も明るくなるでしょうか。鼻、耳、背景などに変えて試しましょう。

In [ ]:
IMAGE = "cat"  # @param ["cat", "astronaut", "coffee", "motorcycle"] {allow-input: true}
X = 0.38  # @param {type:"slider", min:0, max:1, step:0.01}
Y = 0.40  # @param {type:"slider", min:0, max:1, step:0.01}

image = images[IMAGE]
feats = extract(image)
row, col = patch_at(feats, X, Y)
sim = similarity_map(feats, feats["patches"][row, col])  # (縦, 横)。選んだパッチとのコサイン類似度
print(f"選んだパッチ: ({row}, {col})  類似度の範囲: {sim.min():.2f}〜{sim.max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].imshow(image)
axes[0].plot(X * image.width, Y * image.height, "+", color="red", markersize=22, markeredgewidth=3)
axes[0].set_title(f"{IMAGE}: selected patch")
shown = axes[1].imshow(sim.numpy(), cmap="inferno", vmin=0, vmax=1)
axes[1].plot(col, row, "+", color="cyan", markersize=22, markeredgewidth=3)
axes[1].set_title("cosine similarity to the selected patch")
for ax in axes:
    ax.axis("off")
fig.colorbar(shown, ax=axes[1], fraction=0.035)
plt.tight_layout()
plt.show()

### 5-2. 入力の大きさを変える

パッチの大きさは 14×14 画素に決まっているので、入力を大きくすると、パッチの数が増えて、細かいところまで分かれます。トークンの数は入力の短い辺の2乗に比例して増えるので、計算は重くなります（Depth Anything V2 の 5-1 と同じです）。

In [ ]:
IMAGE = "astronaut"  # @param ["cat", "astronaut", "coffee", "motorcycle"] {allow-input: true}

image = images[IMAGE]
sizes = [224, 448, 672]  # 短い辺の画素数（14 の倍数）
fig, axes = plt.subplots(1, len(sizes), figsize=(5 * len(sizes), 4.8))
for ax, size in zip(axes, sizes):
    feats = extract(image, short_side=size)
    gh, gw, d = feats["patches"].shape
    ax.imshow(to_rgb(pca(feats["patches"].reshape(-1, d))[0]).reshape(gh, gw, 3))
    ax.set_title(f"input {feats['input'][2]}x{feats['input'][3]} -> {gh}x{gw} = {gh * gw} patches")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. 活用：パッチの特徴ベクトルを使う

目安：約10分（うち計算 約2分）

### 6-1. 前景を切り出す

DINOv2 の論文では、第1主成分の値が正か負かで分けると、画像の主な物（前景）と背景が分かれる、と報告されています。ここでは、画像の縁のパッチが多く属する側を背景とみなします。ラベルも、切り抜きのための学習もしていません。

In [ ]:
fig, axes = plt.subplots(2, len(images), figsize=(4.2 * len(images), 6.5), squeeze=False)
for k, (name, image) in enumerate(images.items()):
    mask = foreground([extract(image)])[0]  # (縦, 横) の True / False
    big = np.array(Image.fromarray(mask.numpy().astype(np.uint8) * 255).resize(image.size, Image.NEAREST)) > 0
    axes[0, k].imshow(image)
    axes[0, k].set_title(name)
    axes[1, k].imshow(np.where(big[..., None], np.asarray(image.convert("RGB")), 255))
    axes[1, k].set_title(f"foreground: {mask.float().mean():.0%} of patches")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

猫やコーヒーのように主な物がはっきりした画像はよく分かれますが、宇宙飛行士やバイクのように物が多い画像では、うまく分かれないこともあります。

### 6-2. 別の画像で、対応する部分を見つける

第1回のデータセットから、同じ品種の画像を3枚選びます。まず 6-1 と同じ方法で前景を分け（3枚のパッチをまとめて主成分分析します）、次に3枚の前景のパッチだけをまとめて主成分分析し、色で表示します。同じ主成分の空間で色を付けるので、**別の画像でも、同じ部分（目、耳、足など）は同じ色**になります（論文でも同じ図を示しています）。

下の段では、1枚目と2枚目の間で、特徴ベクトルのコサイン類似度がお互いに一番大きいパッチの組（相互最近傍）を、類似度の大きい順に線で結びます。初回は、データセットのダウンロードに1分ほどかかります。

In [ ]:
BREED = "pug"  # @param ["pug", "shiba inu", "persian", "british shorthair", "leonberger", "samoyed", "siamese", "abyssinian"]
LINES = 20  # @param {type:"slider", min:5, max:50, step:5}

pets = load_pets()
label = pets["names"].index(BREED)
picked = [i for i, l in enumerate(pets["test"]["label"]) if l == label][:3]
photos = [pets["test"][i]["image"].convert("RGB") for i in picked]
feats = [extract(p, short_side=336) for p in photos]
masks = foreground(feats)  # 3枚のパッチをまとめて主成分分析し、前景を分ける

# 3枚の前景のパッチをまとめて主成分分析する
fg = torch.cat([f["patches"][m] for f, m in zip(feats, masks)])  # (前景のパッチの合計, 768)
colors = to_rgb(pca(fg)[0])
print(f"前景のパッチ: {[int(m.sum()) for m in masks]} → まとめて {tuple(fg.shape)} → 主成分分析 → {colors.shape}")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
start = 0
for k, (photo, f, m) in enumerate(zip(photos, feats, masks)):
    gh, gw, _ = f["patches"].shape
    rgb = np.ones((gh, gw, 3))
    rgb[m.numpy()] = colors[start:start + int(m.sum())]
    start += int(m.sum())
    axes[0, k].imshow(photo)
    axes[0, k].set_title(f"{BREED} #{k + 1}")
    axes[1, k].imshow(rgb)
    axes[1, k].set_title("foreground parts (shared PCA)")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

# 1枚目と2枚目の対応点（相互最近傍）を線で結ぶ
a, b = [F.normalize(f["patches"][m], dim=-1) for f, m in zip(feats[:2], masks[:2])]
pos_a, pos_b = [torch.nonzero(m) for m in masks[:2]]  # 各前景パッチの (行, 列)
sim = a @ b.T                                          # (1枚目の前景, 2枚目の前景)
best_b, best_a = sim.argmax(dim=1), sim.argmax(dim=0)
mutual = torch.nonzero(best_a[best_b] == torch.arange(len(a)))[:, 0]  # お互いに一番近い組
mutual = mutual[sim[mutual, best_b[mutual]].argsort(descending=True)][:LINES]

height = 336
views = [ImageOps.contain(p, (10000, height)) for p in photos[:2]]
canvas = np.ones((height, views[0].width + views[1].width + 20, 3), dtype=np.uint8) * 255
canvas[:, :views[0].width] = np.asarray(views[0])
canvas[:, views[0].width + 20:] = np.asarray(views[1])
plt.figure(figsize=(14, 5))
plt.imshow(canvas)
for n, i in enumerate(mutual.tolist()):
    (ra, ca), (rb, cb) = pos_a[i].tolist(), pos_b[best_b[i]].tolist()
    ga, gb = feats[0]["patches"].shape, feats[1]["patches"].shape
    xa, ya = (ca + 0.5) / ga[1] * views[0].width, (ra + 0.5) / ga[0] * height
    xb, yb = views[0].width + 20 + (cb + 0.5) / gb[1] * views[1].width, (rb + 0.5) / gb[0] * height
    plt.plot([xa, xb], [ya, yb], "-o", color=plt.cm.rainbow(n / max(1, len(mutual) - 1)), markersize=4, lw=1.2)
plt.title(f"{len(mutual)} mutual nearest neighbor patches (#1 <-> #2)")
plt.axis("off")
plt.show()

### 6-3. 少ない枚数で分類する（k 近傍法）

画像全体のベクトル（CLS トークン）を使って、第1回の37品種を分類します。学習はしません。各品種から `SHOTS` 枚だけ train の画像を選んで「見本」にし、test の画像は、コサイン類似度が一番大きい見本の品種を答えにします（1近傍法）。

- 第1回：train の約2900枚で ResNet-18 を学習して、正解率 約0.83
- CLIP：品種名の文章を使って、学習なしで 約0.87〜0.91

見本が1品種あたり数枚でも、どこまで当てられるでしょうか。test は各品種20枚（計740枚）で測ります。

In [ ]:
SHOTS = 5  # @param [1, 3, 5, 10] {type:"raw"}

pets = load_pets()
generator = torch.Generator().manual_seed(0)


def pick(split, per_class):
    labels = torch.tensor(pets[split]["label"])
    chosen = [torch.nonzero(labels == c)[:, 0] for c in range(len(pets["names"]))]
    return torch.cat([idx[torch.randperm(len(idx), generator=generator)[:per_class]] for idx in chosen])


def cls_vectors(split, indices):
    return F.normalize(torch.stack([extract(pets[split][int(i)]["image"].convert("RGB"), 224)["cls"]
                                    for i in indices]), dim=-1)


start = time.time()
train_idx, test_idx = pick("train", SHOTS), pick("test", 20)
train_vec, test_vec = cls_vectors("train", train_idx), cls_vectors("test", test_idx)  # (37×SHOTS, 768)、(740, 768)
train_lab = torch.tensor(pets["train"]["label"])[train_idx]
test_lab = torch.tensor(pets["test"]["label"])[test_idx]

similarity = test_vec @ train_vec.T            # (740, 37×SHOTS)
preds = train_lab[similarity.argmax(dim=1)]    # 一番近い見本の品種
print(f"見本: 各品種 {SHOTS} 枚 = {len(train_idx)} 枚、test: {len(test_idx)} 枚（{time.time() - start:.0f} 秒）")
print(f"類似度の行列: {tuple(similarity.shape)}")
print(f"正解率: {(preds == test_lab).float().mean():.3f}")

`SHOTS` を 1、3、10 に変えて、正解率がどう変わるか確かめましょう。DINOv2 は品種の名前も文章も知りませんが、画像全体のベクトルが「同じ品種どうしは近い」ように並んでいるので、少ない見本でも分類できます。

## 7. まとめ

目安：約2分

| 分かったこと | 節 |
| --- | --- |
| ViT は、パッチごとに特徴ベクトル（ここでは768次元）を出力する。CNN の特徴マップにあたる | 2-1 |
| DINOv2 は、ラベルも文章も使わない自己教師あり学習で、物とその部分を区別する特徴を身につけた | 2-2 |
| 主成分分析で3次元にして色で見ると、物の部分ごとに色が分かれる | 3. |
| コサイン類似度で、同じ画像や別の画像の中の、似た部分・対応する部分を探せる | 5-1、6-2 |
| 第1主成分で、前景と背景をおおまかに分けられる | 6-1 |
| 画像全体のベクトルを使えば、少ない見本で分類できる | 6-3 |

同じ ViT のパッチの特徴ベクトルが、Depth Anything V2 では奥行きの推定に、ここでは対応点や分類に使われています。研究では、こうした学習済みの特徴を土台にして、自分の課題向けの小さなモデル（分類ヘッドやデコーダ）だけを学習することがよくあります。